In [2]:
import xarray as xr
import tifffile
import numpy as np
import glob, os, gc
import pandas as pd

In [ ]:
# write multi-channel tiff for CANVAS
def prep_canvas(datadir, outpath):
    files = glob.glob(f'{datadir}/10u/counts/*.nc')
    outdir = f'{outpath}/canvas/data/raw_data'

    for f in files:
        name = os.path.splitext(os.path.basename(f))[0].replace('.', '-')
        d = xr.open_dataarray(f)
        avgchannel = d.mean(dim="marker")
        avg_channel_expanded = avgchannel.expand_dims(dim={"marker": ["avg"]})
        d = xr.concat([d, avg_channel_expanded], dim="marker")

        # Save marker values to a text file
        with open(f"{outdir}/image_files/{name}.txt", "w") as txt_file:
            for marker in d.marker.values:
                txt_file.write(f"{marker}\n")

        # Save as a multi-channel TIFF
        arr = d.values  # shape (Y, X, channel)
        arr_tiff = np.transpose(arr, (2, 0, 1))  # shape (channel, Y, X)
        tifffile.imwrite(f"{outdir}/image_files/{name}.tif", arr_tiff)

    infile = glob.glob(f'{outdir}/image_files/*.txt')[0]
    outfile1 = '{outdir}/common_channels.txt'
    outfile2 = '{outdir}/../../configs/preprocess/channels_vis_strength.yaml'

    with open(infile, "r") as infile, open(outfile1, "w") as of1, open(outfile2, "w") as of2:
        for line in infile:
            of1.write(line)
            of2.write(line.strip() + ": 1\n")


In [ ]:
# cells, spots, harmonized spots, pixels?, tiff